# LeetCode 1905 — Count Sub Islands bằng Python

Link đề: https://leetcode.com/problems/count-sub-islands/ · Độ khó: **Medium** · Chủ đề: Array, Depth-First Search, Breadth-First Search, Union Find, Matrix

Skeleton LeetCode:

```python
class Solution:
    def countSubIslands(self, grid1: List[List[int]], grid2: List[List[int]]) -> int:
        ...
```

Lộ trình:
1. **Gán nhãn đảo ở cả hai lưới**, rồi với mỗi đảo của `grid2` kiểm tra mọi ô của nó thuộc **cùng một đảo** của `grid1` (định nghĩa trực tiếp).
2. **DFS lặp trên `grid2`**, trong lúc loang kiểm tra `grid1[i][j] == 1` cho mọi ô — đảo con ⇔ không có ô nào là nước ở `grid1`.
3. **BFS** cùng ý tưởng (hàng đợi thay ngăn xếp).
4. **"Nhấn chìm" trước**: xóa mọi đảo của `grid2` có một ô là nước ở `grid1`, rồi đếm số đảo còn lại.
5. **Union-Find** trên ô đất của `grid2`, mỗi gốc mang cờ "có ô hỏng".

## 1. Mục tiêu học

1. Đếm **thành phần liên thông 4 hướng** trên lưới bằng DFS/BFS **lặp** (lưới tới `500 × 500` — đệ quy sẽ tràn stack).
2. Rút gọn điều kiện "đảo con": đảo `I` của `grid2` là đảo con ⇔ **mọi ô** của `I` là đất trong `grid1` (không cần kiểm "cùng một đảo" vì `I` liên thông).
3. Quan trọng: **không được dừng DFS sớm** khi phát hiện ô hỏng — phải loang hết đảo để không đếm phần còn lại thành đảo mới.
4. Áp dụng kỹ thuật "nhấn chìm" (flood fill xóa) và Union-Find có thông tin phụ ở gốc.
5. Kiểm chứng bằng định nghĩa gán nhãn trên lưới ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho hai ma trận nhị phân `grid1`, `grid2` kích thước `m × n` (`0` = nước, `1` = đất). Một **đảo** là nhóm ô `1` liên thông theo 4 hướng (ngang/dọc). Ô ngoài lưới coi là nước.

Một đảo của `grid2` là **đảo con** nếu tồn tại một đảo của `grid1` **chứa mọi ô** của đảo đó.

Trả về số đảo của `grid2` là đảo con.

**Ví dụ 1:**
```
grid1:            grid2:
1 1 1 0 0         1 1 1 0 0
0 1 1 1 1         0 0 1 1 1
0 0 0 0 0         0 1 0 0 0
1 0 0 0 0         1 0 1 1 0
1 1 0 1 1         0 1 0 1 0
```
→ `3`.

**Ví dụ 2:**
```
grid1:            grid2:
1 0 1 0 1         0 0 0 0 0
1 1 1 1 1         1 1 1 1 1
0 0 0 0 0         0 1 0 1 0
1 1 1 1 1         0 1 0 1 0
1 0 1 0 1         1 0 0 0 1
```
→ `2`.

**Ràng buộc:** `1 ≤ m, n ≤ 500`, các ô là `0` hoặc `1`.

In [1]:
import random
import time
from collections import deque
from typing import List

A1 = [[1,1,1,0,0],[0,1,1,1,1],[0,0,0,0,0],[1,0,0,0,0],[1,1,0,1,1]]
B1 = [[1,1,1,0,0],[0,0,1,1,1],[0,1,0,0,0],[1,0,1,1,0],[0,1,0,1,0]]
A2 = [[1,0,1,0,1],[1,1,1,1,1],[0,0,0,0,0],[1,1,1,1,1],[1,0,1,0,1]]
B2 = [[0,0,0,0,0],[1,1,1,1,1],[0,1,0,1,0],[0,1,0,1,0],[1,0,0,0,1]]
examples = [((A1, B1), 3), ((A2, B2), 2)]

def show(g1, g2):
    for r1, r2 in zip(g1, g2):
        print(" ".join(map(str, r1)), "   ", " ".join(map(str, r2)))

for (g1, g2), want in examples:
    show(g1, g2); print("-> kỳ vọng", want, "\n")

1 1 1 0 0     1 1 1 0 0
0 1 1 1 1     0 0 1 1 1
0 0 0 0 0     0 1 0 0 0
1 0 0 0 0     1 0 1 1 0
1 1 0 1 1     0 1 0 1 0
-> kỳ vọng 3 

1 0 1 0 1     0 0 0 0 0
1 1 1 1 1     1 1 1 1 1
0 0 0 0 0     0 1 0 1 0
1 1 1 1 1     0 1 0 1 0
1 0 1 0 1     1 0 0 0 1
-> kỳ vọng 2 



## 3. Kiến thức nền

### 3.1 DFS lặp đánh dấu một thành phần liên thông

In [2]:
DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))

def component(grid, si, sj, seen):
    """Trả về danh sách ô của đảo chứa (si, sj); đánh dấu seen."""
    m, n = len(grid), len(grid[0])
    st = [(si, sj)]; seen[si][sj] = True; cells = []
    while st:
        i, j = st.pop(); cells.append((i, j))
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and not seen[a][b]:
                seen[a][b] = True; st.append((a, b))
    return cells

seen = [[False] * 5 for _ in range(5)]
print(sorted(component(B1, 0, 0, seen)))

[(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4)]


### 3.2 Gán nhãn mọi đảo

In [3]:
def label_islands(grid):
    m, n = len(grid), len(grid[0])
    lab = [[0] * n for _ in range(m)]; cur = 0
    for i in range(m):
        for j in range(n):
            if grid[i][j] == 1 and lab[i][j] == 0:
                cur += 1; lab[i][j] = cur; st = [(i, j)]
                while st:
                    x, y = st.pop()
                    for dx, dy in DIRS:
                        a, b = x + dx, y + dy
                        if 0 <= a < m and 0 <= b < n and grid[a][b] == 1 and lab[a][b] == 0:
                            lab[a][b] = cur; st.append((a, b))
    return lab, cur

lab, k = label_islands(A1)
print("grid1 có", k, "đảo; nhãn:"); [print(row) for row in lab]

grid1 có 3 đảo; nhãn:
[1, 1, 1, 0, 0]
[0, 1, 1, 1, 1]
[0, 0, 0, 0, 0]
[2, 0, 0, 0, 0]
[2, 2, 0, 3, 3]


[None, None, None, None, None]

### 3.3 Union-Find với thông tin ở gốc

Mỗi tập lưu thêm cờ `bad[root]`; khi hợp hai tập, cờ mới = OR của hai cờ.

In [4]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n)); self.bad = [False] * n
    def find(self, x):
        root = x
        while self.p[root] != root: root = self.p[root]
        while self.p[x] != root: self.p[x], x = root, self.p[x]
        return root
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[ra] = rb; self.bad[rb] = self.bad[rb] or self.bad[ra]

d = DSU(4); d.bad[1] = True; d.union(0, 1); d.union(2, 3)
print([d.find(i) for i in range(4)], [d.bad[d.find(i)] for i in range(4)])

[1, 1, 3, 3] [True, True, False, False]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: các đảo của `grid2` và việc chúng có "nằm trên đất" của `grid1` không:

| đảo của grid2 | các ô | ô nào là nước ở grid1? | đảo con? |
|---|---|---|---|
| A | (0,0),(0,1),(0,2),(1,2),(1,3),(1,4) | không | ✓ |
| B | (2,1) | (2,1) là nước | ✗ |
| C | (3,0) | không | ✓ |
| D | (3,2),(3,3),(4,3) | (3,2),(3,3) | ✗ |
| E | (4,1) | không | ✓ |

⇒ `3` đảo con. ✓

**Vì sao chỉ cần kiểm "mọi ô là đất ở grid1"?** Nếu mọi ô của đảo `I` (liên thông trong `grid2`) đều là đất ở `grid1`, thì các ô đó cũng **liên thông trong `grid1`** (dùng cùng các cạnh kề) ⇒ cùng nằm trong **một** đảo của `grid1`.

In [5]:
seen = [[False] * 5 for _ in range(5)]
for i in range(5):
    for j in range(5):
        if B1[i][j] == 1 and not seen[i][j]:
            cells = component(B1, i, j, seen)
            water = [c for c in cells if A1[c[0]][c[1]] == 0]
            print(f"đảo {sorted(cells)}: ô nước ở grid1 = {water} -> {'đảo con' if not water else 'không'}")

đảo [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4)]: ô nước ở grid1 = [] -> đảo con
đảo [(2, 1)]: ô nước ở grid1 = [(2, 1)] -> không
đảo [(3, 0)]: ô nước ở grid1 = [] -> đảo con
đảo [(3, 2), (3, 3), (4, 3)]: ô nước ở grid1 = [(3, 2), (3, 3)] -> không
đảo [(4, 1)]: ô nước ở grid1 = [] -> đảo con


## 5. Cách 1 — Brute Force / baseline: gán nhãn cả hai lưới

**Ý tưởng (theo định nghĩa):** gán nhãn đảo của `grid1`. Với mỗi đảo của `grid2`, lấy tập nhãn `grid1` của các ô (nhãn 0 = nước). Đảo con ⇔ tập nhãn là `{L}` với `L ≠ 0`.

In [6]:
def count_sub_labels(grid1, grid2):
    lab1, _ = label_islands(grid1)
    m, n = len(grid2), len(grid2[0])
    seen = [[False] * n for _ in range(m)]
    cnt = 0
    for i in range(m):
        for j in range(n):
            if grid2[i][j] == 1 and not seen[i][j]:
                cells = component(grid2, i, j, seen)
                labels = {lab1[a][b] for a, b in cells}
                if len(labels) == 1 and 0 not in labels:
                    cnt += 1
    return cnt

for (g1, g2), want in examples:
    print(count_sub_labels(g1, g2), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** gán nhãn `O(m n)` + duyệt `grid2` `O(m n)` + tạo tập nhãn `O(kích thước đảo)` ⇒ `O(m n)` thời gian, `O(m n)` bộ nhớ (nhãn + `seen` + danh sách ô). Đã tối ưu về bậc, nhưng làm hai lượt loang và lưu nhiều.

**Nút thắt:** không cần nhãn của `grid1`; chỉ cần biết ô có là đất không (mục 4).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** đảo `I` của `grid2` là đảo con ⇔ `∀ (i, j) ∈ I : grid1[i][j] = 1`.

**Quy tắc khi loang một đảo:** giữ cờ `ok = True`; gặp ô có `grid1 = 0` thì `ok = False` nhưng **vẫn loang tiếp** cho hết đảo (đánh dấu đã thăm). Hết đảo, nếu `ok` thì đếm.

**Biến thể "nhấn chìm":** lượt 1 — với mọi ô `grid2 = 1` và `grid1 = 0`, loang xóa cả đảo chứa nó; lượt 2 — đếm số đảo còn lại của `grid2`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS lặp với cờ `ok`

In [7]:
def count_sub_dfs(grid1, grid2):
    m, n = len(grid2), len(grid2[0])
    seen = [[False] * n for _ in range(m)]
    cnt = 0
    for si in range(m):
        for sj in range(n):
            if grid2[si][sj] == 1 and not seen[si][sj]:
                ok = True
                seen[si][sj] = True
                st = [(si, sj)]
                while st:
                    i, j = st.pop()
                    if grid1[i][j] == 0:
                        ok = False                    # KHÔNG dừng — vẫn loang hết đảo
                    for di, dj in DIRS:
                        a, b = i + di, j + dj
                        if 0 <= a < m and 0 <= b < n and grid2[a][b] == 1 and not seen[a][b]:
                            seen[a][b] = True
                            st.append((a, b))
                cnt += ok
    return cnt

for (g1, g2), want in examples:
    print(count_sub_dfs(g1, g2), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** Time `O(m n)` (mỗi ô vào ngăn xếp một lần), Space `O(m n)` (`seen` + ngăn xếp).

### 7.2 Cách 3 — BFS, đánh dấu bằng cách sửa bản sao `grid2`

Không cần mảng `seen`: ghi `0` vào bản sao `grid2` khi thăm. (Không sửa input của người gọi.)

In [8]:
def count_sub_bfs(grid1, grid2):
    g = [row[:] for row in grid2]
    m, n = len(g), len(g[0])
    cnt = 0
    for si in range(m):
        for sj in range(n):
            if g[si][sj] == 1:
                ok = True
                g[si][sj] = 0
                q = deque([(si, sj)])
                while q:
                    i, j = q.popleft()
                    ok &= grid1[i][j] == 1
                    for di, dj in DIRS:
                        a, b = i + di, j + dj
                        if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                            g[a][b] = 0
                            q.append((a, b))
                cnt += ok
    return cnt

for (g1, g2), want in examples:
    print(count_sub_bfs(g1, g2), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


Code nộp LeetCode của Cách 3: xem mục 9.

**Độ phức tạp:** Time `O(m n)`, Space `O(m n)` (bản sao + hàng đợi `O(min(m, n))` điển hình).

### 7.3 Cách 4 — "Nhấn chìm" các đảo hỏng trước, rồi đếm

Lượt 1: với mọi ô `grid2 = 1` mà `grid1 = 0`, loang và xóa cả đảo (đảo đó chắc chắn không phải đảo con). Lượt 2: đếm số đảo còn lại — tất cả đều là đảo con.

In [9]:
def sink(g, si, sj):
    m, n = len(g), len(g[0])
    g[si][sj] = 0; st = [(si, sj)]
    while st:
        i, j = st.pop()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                g[a][b] = 0; st.append((a, b))

def count_sub_sink(grid1, grid2):
    g = [row[:] for row in grid2]
    m, n = len(g), len(g[0])
    for i in range(m):
        for j in range(n):
            if g[i][j] == 1 and grid1[i][j] == 0:
                sink(g, i, j)
    cnt = 0
    for i in range(m):
        for j in range(n):
            if g[i][j] == 1:
                sink(g, i, j); cnt += 1
    return cnt

for (g1, g2), want in examples:
    print(count_sub_sink(g1, g2), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** mỗi ô bị xóa tối đa một lần ⇒ `O(m n)`, Space `O(m n)`. Ưu điểm: logic đếm ở lượt 2 là bài "Number of Islands" chuẩn — ít chỗ sai.

### 7.4 Cách 5 — Union-Find trên ô đất của `grid2`

Mỗi ô đất `(i, j)` là một phần tử `i·n + j`; `bad[x] = (grid1[i][j] == 0)`. Hợp với ô đất kề phải/dưới. Đáp án = số gốc có `bad = False`.

In [10]:
def count_sub_dsu(grid1, grid2):
    m, n = len(grid2), len(grid2[0])
    d = DSU(m * n)
    for i in range(m):
        for j in range(n):
            if grid2[i][j] == 1:
                d.bad[i * n + j] = grid1[i][j] == 0
    for i in range(m):
        for j in range(n):
            if grid2[i][j] == 1:
                if i + 1 < m and grid2[i + 1][j] == 1:
                    d.union(i * n + j, (i + 1) * n + j)
                if j + 1 < n and grid2[i][j + 1] == 1:
                    d.union(i * n + j, i * n + j + 1)
    roots = {d.find(i * n + j) for i in range(m) for j in range(n) if grid2[i][j] == 1}
    return sum(1 for r in roots if not d.bad[r])

for (g1, g2), want in examples:
    print(count_sub_dsu(g1, g2), "kỳ vọng", want)

3 kỳ vọng 3
2 kỳ vọng 2


**Độ phức tạp:** `O(m n · α(m n))`, Space `O(m n)`.

## 8. TRACE từng bước

Trace Cách 2 trên ví dụ 1 — in từng đảo: ô bắt đầu, các ô thăm, ô làm hỏng:

In [11]:
def trace_dfs(grid1, grid2):
    m, n = len(grid2), len(grid2[0]); seen = [[False] * n for _ in range(m)]; cnt = 0
    for si in range(m):
        for sj in range(n):
            if grid2[si][sj] == 1 and not seen[si][sj]:
                ok = True; seen[si][sj] = True; st = [(si, sj)]; order = []; bad = []
                while st:
                    i, j = st.pop(); order.append((i, j))
                    if grid1[i][j] == 0: ok = False; bad.append((i, j))
                    for di, dj in DIRS:
                        a, b = i + di, j + dj
                        if 0 <= a < m and 0 <= b < n and grid2[a][b] == 1 and not seen[a][b]:
                            seen[a][b] = True; st.append((a, b))
                cnt += ok
                print(f"đảo bắt đầu {(si, sj)}: thăm {order}; ô hỏng {bad or '-'} -> {'ĐẾM' if ok else 'bỏ'} (tổng {cnt})")
    return cnt

trace_dfs(A1, B1)

đảo bắt đầu (0, 0): thăm [(0, 0), (0, 1), (0, 2), (1, 2), (1, 3), (1, 4)]; ô hỏng - -> ĐẾM (tổng 1)
đảo bắt đầu (2, 1): thăm [(2, 1)]; ô hỏng [(2, 1)] -> bỏ (tổng 1)
đảo bắt đầu (3, 0): thăm [(3, 0)]; ô hỏng - -> ĐẾM (tổng 2)
đảo bắt đầu (3, 2): thăm [(3, 2), (3, 3), (4, 3)]; ô hỏng [(3, 2), (3, 3)] -> bỏ (tổng 2)
đảo bắt đầu (4, 1): thăm [(4, 1)]; ô hỏng - -> ĐẾM (tổng 3)


3

In [12]:
def trace_sink(grid1, grid2):
    g = [row[:] for row in grid2]; m, n = len(g), len(g[0])
    for i in range(m):
        for j in range(n):
            if g[i][j] == 1 and grid1[i][j] == 0:
                print(f"ô ({i},{j}) là đất ở grid2 nhưng nước ở grid1 -> nhấn chìm cả đảo"); sink(g, i, j)
    print("grid2 sau khi nhấn chìm:"); [print(" ".join(map(str, r))) for r in g]
    cnt = 0
    for i in range(m):
        for j in range(n):
            if g[i][j] == 1: sink(g, i, j); cnt += 1
    print("số đảo còn lại =", cnt)

trace_sink(A2, B2)

ô (2,1) là đất ở grid2 nhưng nước ở grid1 -> nhấn chìm cả đảo
grid2 sau khi nhấn chìm:
0 0 0 0 0
0 0 0 0 0
0 0 0 0 0
0 0 0 0 0
1 0 0 0 1
số đảo còn lại = 2


Bảng tóm tắt ví dụ 2: đảo hàng 1 của `grid2` (`(1,0)…(1,4)` nối với `(2,1),(3,1)` và `(2,3),(3,3)`) chứa `(2,1)` là nước ở `grid1` ⇒ bị nhấn chìm. Còn lại hai đảo một ô `(4,0)`, `(4,4)` — đều là đất ở `grid1` ⇒ `2`.

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def countSubIslands(self, grid1: List[List[int]], grid2: List[List[int]]) -> int:
        g = [row[:] for row in grid2]
        m, n = len(g), len(g[0])
        cnt = 0
        for si in range(m):
            for sj in range(n):
                if g[si][sj] == 1:
                    ok = True
                    g[si][sj] = 0
                    q = deque([(si, sj)])
                    while q:
                        i, j = q.popleft()
                        if grid1[i][j] == 0:
                            ok = False
                        for a, b in ((i + 1, j), (i - 1, j), (i, j + 1), (i, j - 1)):
                            if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                                g[a][b] = 0
                                q.append((a, b))
                    if ok:
                        cnt += 1
        return cnt

sol = Solution()
for (g1, g2), want in examples:
    got = sol.countSubIslands(g1, g2)
    print(got, "OK" if got == want else "SAI")

3 OK
2 OK


## 10. Vì sao thuật toán đúng

**Bổ đề.** Đảo `I` của `grid2` là đảo con ⇔ mọi ô của `I` là đất trong `grid1`.
- (⇒) Nếu một đảo `J` của `grid1` chứa `I` thì mọi ô của `I` là đất của `grid1`.
- (⇐) Giả sử mọi ô của `I` là đất trong `grid1`. Vì `I` liên thông trong `grid2` (qua các cặp ô kề đều thuộc `I`), các cặp ô kề đó cũng là cặp đất kề trong `grid1` ⇒ `I` liên thông trong `grid1` ⇒ nằm trọn trong một đảo `J` của `grid1`. ✓

**Cách 2/3.** Mỗi lần khởi động BFS/DFS từ một ô đất chưa thăm, ta thăm **đúng** một đảo của `grid2` (thành phần liên thông). Cờ `ok` = AND của `grid1[i][j] == 1` trên mọi ô ⇒ đúng điều kiện bổ đề. Vì loang **hết** đảo trước khi đếm, không ô nào của đảo bị "sót" để rồi khởi động một lượt loang mới.

**Cách 4.** Lượt 1 xóa đúng các đảo có ô hỏng (loang từ ô hỏng xóa cả đảo chứa nó). Đảo không có ô hỏng không bị chạm. Lượt 2 đếm đúng số đảo còn lại.

**Cách 5.** Union-Find gộp đúng các thành phần liên thông; cờ `bad` của gốc = OR của mọi ô trong thành phần.

## 11. Phân tích Time / Space

Gọi `N = m · n ≤ 2.5·10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Gán nhãn hai lưới | `O(N)` | `O(N)` | theo định nghĩa |
| 2. DFS lặp + cờ | `O(N)` | `O(N)` | một lượt |
| 3. BFS + sửa bản sao | `O(N)` | `O(N)` | **nộp bài** |
| 4. Nhấn chìm rồi đếm | `O(N)` | `O(N)` | ít lỗi |
| 5. Union-Find | `O(N α(N))` | `O(N)` | mở rộng truy vấn động |

In [14]:
def random_pair(m, n, rng, p1=0.7, p2=0.5):
    g1 = [[1 if rng.random() < p1 else 0 for _ in range(n)] for _ in range(m)]
    g2 = [[1 if rng.random() < p2 else 0 for _ in range(n)] for _ in range(m)]
    return g1, g2

rng = random.Random(1)
big = random_pair(500, 500, rng)
full = ([[1] * 500 for _ in range(500)], [[1] * 500 for _ in range(500)])     # một đảo khổng lồ
for name, fn in [("labels", count_sub_labels), ("dfs", count_sub_dfs), ("bfs", count_sub_bfs), ("sink", count_sub_sink),
                 ("dsu", count_sub_dsu), ("submit", Solution().countSubIslands)]:
    t0 = time.perf_counter(); a = fn(*big); b = fn(*full)
    print(f"{name:6s}: {a} {b}  {time.perf_counter() - t0:.2f}s")

labels: 7554 1  0.49s


dfs   : 7554 1  0.21s
bfs   : 7554 1  0.19s


sink  : 7554 1  0.19s


dsu   : 7554 1  0.39s
submit: 7554 1  0.21s


## 12. Bộ test

In [15]:
fns = [count_sub_dfs, count_sub_bfs, count_sub_sink, count_sub_dsu, lambda a, b: Solution().countSubIslands(a, b)]

for (g1, g2), want in examples:
    assert count_sub_labels(g1, g2) == want
    for fn in fns:
        assert fn(g1, g2) == want

edge = [
    (([[1]], [[1]]), 1), (([[0]], [[1]]), 0), (([[1]], [[0]]), 0),
    (([[1, 0, 1]], [[1, 1, 1]]), 0),                 # đảo grid2 vắt qua nước của grid1
    (([[1, 1, 1]], [[1, 0, 1]]), 2),                 # hai đảo nhỏ trong cùng một đảo lớn
    (([[1, 0], [0, 1]], [[1, 0], [0, 1]]), 2),       # chéo không liên thông
    (([[0, 0], [0, 0]], [[0, 0], [0, 0]]), 0),
]
for (g1, g2), want in edge:
    assert count_sub_labels(g1, g2) == want, (g1, g2)
    for fn in fns:
        assert fn(g1, g2) == want, (fn, g1, g2)

rng = random.Random(2024)
for _ in range(800):
    m, n = rng.randint(1, 9), rng.randint(1, 9)
    g1, g2 = random_pair(m, n, rng, rng.random(), rng.random())
    want = count_sub_labels(g1, g2)
    snap = [row[:] for row in g2]
    for fn in fns:
        assert fn(g1, g2) == want, (fn, g1, g2)
    assert g2 == snap                               # không sửa input
print("ví dụ + edge + 800 cặp lưới ngẫu nhiên khớp định nghĩa; input không bị sửa")

ví dụ + edge + 800 cặp lưới ngẫu nhiên khớp định nghĩa; input không bị sửa


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dừng DFS ngay khi gặp ô hỏng

In [16]:
def wrong_early_stop(grid1, grid2):
    m, n = len(grid2), len(grid2[0]); seen = [[False] * n for _ in range(m)]; cnt = 0
    for si in range(m):
        for sj in range(n):
            if grid2[si][sj] == 1 and not seen[si][sj]:
                ok = True; seen[si][sj] = True; st = [(si, sj)]
                while st:
                    i, j = st.pop()
                    if grid1[i][j] == 0:
                        ok = False; break                 # SAI: phần còn lại của đảo chưa được đánh dấu
                    for di, dj in DIRS:
                        a, b = i + di, j + dj
                        if 0 <= a < m and 0 <= b < n and grid2[a][b] == 1 and not seen[a][b]:
                            seen[a][b] = True; st.append((a, b))
                cnt += ok
    return cnt

for (g1, g2), want in examples:
    print("sai =", wrong_early_stop(g1, g2), "| đúng =", want)

sai = 4 | đúng = 3
sai = 4 | đúng = 2


Phần đảo chưa thăm sẽ được khởi động lại ở vòng ngoài như một "đảo mới" — có thể bị đếm nhầm là đảo con. **Sửa:** chỉ đặt `ok = False` và **loang tiếp**.

### Lỗi 2 — Kiểm tra "ô bắt đầu" thay vì mọi ô

In [17]:
def wrong_start_only(grid1, grid2):
    m, n = len(grid2), len(grid2[0]); seen = [[False] * n for _ in range(m)]; cnt = 0
    for i in range(m):
        for j in range(n):
            if grid2[i][j] == 1 and not seen[i][j]:
                component(grid2, i, j, seen)
                cnt += grid1[i][j] == 1                       # SAI: chỉ kiểm một ô
    return cnt

for (g1, g2), want in examples:
    print("sai =", wrong_start_only(g1, g2), "| đúng =", want)

sai = 3 | đúng = 3
sai = 3 | đúng = 2


**Sửa:** AND điều kiện trên **mọi** ô của đảo.

### Lỗi 3 — Đệ quy DFS trên lưới lớn

In [18]:
import sys

def dfs_rec(g, i, j):
    if not (0 <= i < len(g) and 0 <= j < len(g[0])) or g[i][j] != 1:
        return
    g[i][j] = 0
    for di, dj in DIRS:
        dfs_rec(g, i + di, j + dj)

g = [[1] * 60 for _ in range(60)]
try:
    dfs_rec(g, 0, 0); print("chạy được")
except RecursionError as e:
    print("RecursionError trên lưới 60x60 toàn đất:", str(e)[:40], f"(giới hạn {sys.getrecursionlimit()})")

RecursionError trên lưới 60x60 toàn đất: maximum recursion depth exceeded (giới hạn 1000)


Lưới `500 × 500` toàn đất cần độ sâu đệ quy tới `2.5·10⁵`. **Sửa:** DFS/BFS lặp.

### Lỗi 4 — Coi láng giềng **chéo** là liên thông

In [19]:
DIRS8 = DIRS + ((1, 1), (1, -1), (-1, 1), (-1, -1))

def wrong_diagonal(grid1, grid2):
    m, n = len(grid2), len(grid2[0]); seen = [[False] * n for _ in range(m)]; cnt = 0
    for si in range(m):
        for sj in range(n):
            if grid2[si][sj] == 1 and not seen[si][sj]:
                ok = True; seen[si][sj] = True; st = [(si, sj)]
                while st:
                    i, j = st.pop(); ok &= grid1[i][j] == 1
                    for di, dj in DIRS8:                       # SAI: đề dùng 4 hướng
                        a, b = i + di, j + dj
                        if 0 <= a < m and 0 <= b < n and grid2[a][b] == 1 and not seen[a][b]:
                            seen[a][b] = True; st.append((a, b))
                cnt += ok
    return cnt

for (g1, g2), want in examples + [(([[1, 0], [0, 1]], [[1, 0], [0, 1]]), 2)]:
    print("sai =", wrong_diagonal(g1, g2), "| đúng =", want)

sai = 0 | đúng = 3
sai = 0 | đúng = 2
sai = 1 | đúng = 2


**Sửa:** chỉ 4 hướng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm thành phần liên thông thỏa điều kiện trên **mọi** ô" ⇒ loang hết thành phần, giữ cờ AND, **không dừng sớm**.
- Điều kiện "nằm trong một thành phần của lưới khác" thường rút gọn được thành điều kiện **từng ô** (do tính liên thông).
- Kỹ thuật "nhấn chìm trước" biến bài thành Number of Islands chuẩn.
- Python + lưới lớn ⇒ DFS/BFS lặp; sửa trên **bản sao** nếu không muốn đổi input.
- Họ bài: 200 *Number of Islands*, 1254 *Number of Closed Islands*, 1020 *Number of Enclaves*, 695 *Max Area of Island*.

## 15. Biến thể của bài

1. **Tổng diện tích các đảo con:** cộng kích thước đảo khi `ok`.
2. **Đảo con phải trùng khớp hoàn toàn** với một đảo của `grid1` (bằng nhau): thêm điều kiện kích thước đảo bằng kích thước đảo `grid1` chứa nó.
3. **1254 Number of Closed Islands:** đảo không chạm biên — cùng khung cờ AND (`ok &= không ở biên`).
4. **Truy vấn động** (lật ô của `grid1` từ đất thành nước): Union-Find ngược thời gian.

In [20]:
def sub_island_area(grid1, grid2):
    """Biến thể 1."""
    g = [row[:] for row in grid2]; m, n = len(g), len(g[0]); total = 0
    for si in range(m):
        for sj in range(n):
            if g[si][sj] == 1:
                ok = True; size = 0; g[si][sj] = 0; st = [(si, sj)]
                while st:
                    i, j = st.pop(); size += 1; ok &= grid1[i][j] == 1
                    for di, dj in DIRS:
                        a, b = i + di, j + dj
                        if 0 <= a < m and 0 <= b < n and g[a][b] == 1:
                            g[a][b] = 0; st.append((a, b))
                if ok: total += size
    return total

def exact_islands(grid1, grid2):
    """Biến thể 2: đảo của grid2 trùng khít với một đảo của grid1."""
    lab1, k = label_islands(grid1)
    size1 = [0] * (k + 1)
    for row in lab1:
        for v in row: size1[v] += 1
    m, n = len(grid2), len(grid2[0]); seen = [[False] * n for _ in range(m)]; cnt = 0
    for i in range(m):
        for j in range(n):
            if grid2[i][j] == 1 and not seen[i][j]:
                cells = component(grid2, i, j, seen); labs = {lab1[a][b] for a, b in cells}
                if len(labs) == 1 and 0 not in labs and size1[labs.pop()] == len(cells):
                    cnt += 1
    return cnt

print("diện tích đảo con ví dụ 1:", sub_island_area(A1, B1), "| số đảo trùng khít:", exact_islands(A1, B1), exact_islands(A1, A1))

diện tích đảo con ví dụ 1: 8 | số đảo trùng khít: 0 3


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Gán nhãn | đúng định nghĩa | `O(N)` | `O(N)` | kiểm chứng |
| 2. DFS + cờ | AND trên mọi ô | `O(N)` | `O(N)` | gọn |
| 3. BFS + bản sao | không cần `seen` | `O(N)` | `O(N)` | **nộp bài** |
| 4. Nhấn chìm | xóa đảo hỏng trước | `O(N)` | `O(N)` | ít lỗi logic |
| 5. Union-Find | cờ ở gốc | `O(N α)` | `O(N)` | mở rộng động |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `grid1 = [[1,1],[1,1]]`, `grid2 = [[1,0],[0,1]]`? Và `grid1 = [[1,0],[1,1]]`, `grid2 = [[1,1],[0,1]]`?

**Bài 2 (tính tay):** với ví dụ 2, liệt kê các đảo của `grid2` và ô hỏng của từng đảo.

**Bài 3 (điền TODO):** hoàn thiện `bad_cells(grid1, grid2)` trả về danh sách ô `(i, j)` là đất ở `grid2` nhưng nước ở `grid1`.

**Bài 4 (tự viết lại):** viết `count_sub_one_pass_bool(grid1, grid2)` dùng mảng `bytearray` làm `seen` và ngăn xếp phẳng (lưu `i * n + j`) để tối ưu hằng số; đo thời gian trên lưới `500 × 500`.

**Bài 5 (mini challenge):** biến thể 4 — cho `grid1`, `grid2` và danh sách thao tác "biến ô `(i, j)` của `grid1` thành nước". Sau **mỗi** thao tác, trả về số đảo con. Viết lời giải đơn giản `O(q · N)` và kiểm chứng; (thử thách: `O((N + q) α)` bằng Union-Find xử lý ngược). (Biến thể tự đặt.)

In [21]:
def bad_cells(grid1, grid2):
    # TODO: [(i, j) for ... if grid2[i][j] == 1 and grid1[i][j] == 0]
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) Hai đảo một ô `(0,0)`, `(1,1)` đều trên đất ⇒ `2`. (2) Lưới thứ hai: đảo `{(0,0),(0,1),(1,1)}` có `(0,1)` là nước ở `grid1`. (3) ⇒ `0`.

**Bài 2:** (1) Đảo lớn gồm hàng 1 và `(2,1),(3,1),(2,3),(3,3)`. (2) Ô hỏng: `(2,1)`, `(2,3)` (hàng 2 của `grid1` toàn nước). (3) Hai đảo `(4,0)`, `(4,4)` không hỏng.

**Bài 3:** (1) Hai vòng lặp. (2) Điều kiện kép. (3) Dùng để kiểm tra Cách 4 (mỗi ô hỏng ⇒ đảo chứa nó bị nhấn chìm).

**Bài 4:** (1) `seen = bytearray(m * n)`. (2) Láng giềng của `p = i·n + j`: `p ± n`, `p ± 1` (cẩn thận biên cột). (3) So thời gian với Cách 2.

**Bài 5:** (1) Đơn giản: sau mỗi thao tác chạy lại Cách 3. (2) Kiểm chứng bằng Cách 1. (3) Nâng cao: xử lý ngược, mỗi ô "hỏng" đóng góp vào cờ `bad` của gốc; khi đi ngược, ô hết hỏng ⇒ cần **đếm** số ô hỏng mỗi gốc thay vì cờ.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert Solution().countSubIslands([[1, 1], [1, 1]], [[1, 0], [0, 1]]) == 2
assert Solution().countSubIslands([[1, 0], [1, 1]], [[1, 1], [0, 1]]) == 0

# Bài 2
trace_dfs(A2, B2)

# Bài 3
def bad_cells_ans(grid1, grid2):
    return [(i, j) for i in range(len(grid2)) for j in range(len(grid2[0])) if grid2[i][j] == 1 and grid1[i][j] == 0]
print("ô hỏng ví dụ 2:", bad_cells_ans(A2, B2))

# Bài 4
def count_sub_flat(grid1, grid2):
    m, n = len(grid2), len(grid2[0]); N = m * n
    land2 = bytearray(v for row in grid2 for v in row); land1 = bytearray(v for row in grid1 for v in row)
    seen = bytearray(N); cnt = 0
    for s in range(N):
        if land2[s] and not seen[s]:
            ok = True; seen[s] = 1; st = [s]
            while st:
                p = st.pop()
                if not land1[p]: ok = False
                j = p % n
                for q in (p - n if p >= n else -1, p + n if p + n < N else -1,
                          p - 1 if j > 0 else -1, p + 1 if j < n - 1 else -1):
                    if q >= 0 and land2[q] and not seen[q]:
                        seen[q] = 1; st.append(q)
            cnt += ok
    return cnt

rng = random.Random(281)
for _ in range(300):
    g1, g2 = random_pair(rng.randint(1, 9), rng.randint(1, 9), rng, rng.random(), rng.random())
    assert count_sub_flat(g1, g2) == count_sub_labels(g1, g2)
t0 = time.perf_counter(); r = count_sub_flat(*big); t1 = time.perf_counter(); r2 = count_sub_dfs(*big); t2 = time.perf_counter()
assert r == r2
print(f"Bài 4: phẳng {t1 - t0:.2f}s | DFS thường {t2 - t1:.2f}s")
print("Bài 1-4: OK")

đảo bắt đầu (1, 0): thăm [(1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (2, 3), (3, 3), (2, 1), (3, 1)]; ô hỏng [(2, 3), (2, 1)] -> bỏ (tổng 0)
đảo bắt đầu (4, 0): thăm [(4, 0)]; ô hỏng - -> ĐẾM (tổng 1)
đảo bắt đầu (4, 4): thăm [(4, 4)]; ô hỏng - -> ĐẾM (tổng 2)
ô hỏng ví dụ 2: [(2, 1), (2, 3)]
Bài 4: phẳng 0.07s | DFS thường 0.08s
Bài 1-4: OK


In [23]:
# Bài 5 — cập nhật động (lời giải đơn giản O(q·N), kiểm chứng bằng định nghĩa)
def dynamic_sub_islands(grid1, grid2, ops):
    g1 = [row[:] for row in grid1]; out = []
    for i, j in ops:
        g1[i][j] = 0
        out.append(Solution().countSubIslands(g1, grid2))
    return out

def dynamic_reverse_dsu(grid1, grid2, ops):
    """Thử thách: xử lý NGƯỢC thời gian; mỗi gốc lưu SỐ ô hỏng."""
    m, n = len(grid2), len(grid2[0])
    g1 = [row[:] for row in grid1]
    for i, j in ops: g1[i][j] = 0
    d = DSU(m * n); badcnt = [0] * (m * n)
    for i in range(m):
        for j in range(n):
            if grid2[i][j] == 1: badcnt[i * n + j] = int(g1[i][j] == 0)
    def union(a, b):
        ra, rb = d.find(a), d.find(b)
        if ra != rb: d.p[ra] = rb; badcnt[rb] += badcnt[ra]
    for i in range(m):
        for j in range(n):
            if grid2[i][j] == 1:
                if i + 1 < m and grid2[i + 1][j] == 1: union(i * n + j, (i + 1) * n + j)
                if j + 1 < n and grid2[i][j + 1] == 1: union(i * n + j, i * n + j + 1)
    roots = {d.find(i * n + j) for i in range(m) for j in range(n) if grid2[i][j] == 1}
    good = sum(1 for r in roots if badcnt[r] == 0)
    res = [0] * len(ops)
    for t in range(len(ops) - 1, -1, -1):
        res[t] = good
        i, j = ops[t]
        if grid1[i][j] == 1 and all(ops[s] != (i, j) for s in range(t)):   # trước thao tác t ô này là đất
            if grid2[i][j] == 1:
                r = d.find(i * n + j)
                badcnt[r] -= 1
                if badcnt[r] == 0: good += 1
    return res

rng = random.Random(282)
for _ in range(200):
    m, n = rng.randint(1, 6), rng.randint(1, 6)
    g1, g2 = random_pair(m, n, rng, 0.8, 0.5)
    cells = [(i, j) for i in range(m) for j in range(n)]
    ops = rng.sample(cells, rng.randint(1, len(cells)))
    a = dynamic_sub_islands(g1, g2, ops)
    assert a == dynamic_reverse_dsu(g1, g2, ops), (g1, g2, ops)
print("Bài 5: cập nhật động — DSU ngược khớp tính lại từ đầu; ví dụ:", dynamic_sub_islands(A1, B1, [(0, 0), (3, 0), (4, 1)]))

Bài 5: cập nhật động — DSU ngược khớp tính lại từ đầu; ví dụ: [2, 1, 0]


## 20. Template nhớ nhanh + Checklist

```python
# Đếm đảo của grid2 mà MỌI ô đều là đất trong grid1
g = [row[:] for row in grid2]                    # bản sao để đánh dấu
for each (si, sj) with g[si][sj] == 1:
    ok = True; g[si][sj] = 0; q = deque([(si, sj)])
    while q:
        i, j = q.popleft()
        if grid1[i][j] == 0: ok = False          # KHÔNG dừng — loang hết đảo
        for (a, b) in 4 láng giềng trong lưới:
            if g[a][b] == 1: g[a][b] = 0; q.append((a, b))
    cnt += ok
```

**Checklist trước khi submit:**
- [ ] Loang **hết** đảo dù đã phát hiện ô hỏng.
- [ ] Kiểm tra **mọi** ô của đảo, không chỉ ô bắt đầu.
- [ ] Chỉ 4 hướng.
- [ ] DFS/BFS lặp (lưới tới `500 × 500`).
- [ ] Không sửa input (hoặc chấp nhận sửa nếu đề cho phép).